# DSCI_575 Amazon Product Query Assistant
Alan Liu, Zhihao Xie

## Download Data

In [11]:
import os
import requests
from pathlib import Path

# Asked Gemini how to download data from link to jsonl.gz
def download_data(url, file_name):
    data_folder = Path("../data/raw")
    data_folder.mkdir(parents=True, exist_ok=True)

    save_path = data_folder / file_name

    print(f"Downloading to: {save_path}...")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print("Download complete!")

In [2]:
# download review data
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Sports_and_Outdoors.jsonl.gz"
file_name = "Sports_and_Outdoors.jsonl.gz"
download_data(url, file_name)

# download meta data
url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Sports_and_Outdoors.jsonl.gz"
file_name = "meta_Sports_and_Outdoors.jsonl.gz"
download_data(url, file_name)

Download complete!
Download complete!


In [12]:
url = "https://ontario.ca/v1/files/fuel-prices/canadianpumppricesall.csv"
file_name = "file.csv"
download_data(url, file_name)

Download complete!


## Load Data
- an overview of the dataset (fields, size, example records)
selection and justification of fields for retrieval
description of text preprocessing decisions

### An overview of the dataset

In [29]:
def count_num_of_records(file_path):
    number_of_records = 0
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            # Asked Gemini: how to print JSON in format
            product_key = data.get("asin")
            if product_key:
                number_of_records += 1
    return number_of_records

In [31]:
meta_path = data_folder / "meta_Sports_and_Outdoors.jsonl.gz"
print(f"Number of products: {count_num_of_records(meta_path)}")

Number of products: 0


### Inspection of sample records:
Looks like the title of the product is in `meta_Sports_and_Outdoors.jsonl.gz`

In [19]:
import gzip
import json
from collections import Counter

### Examine Meta Data
It looks like the following fields are useful:
- title
- average_rating
- features (after concating all the elements into one string)
- description
- price
- categories
- parent_asin

In [32]:
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        print(data.keys())
        print(json.dumps(data, indent=4))
        print(data['videos'])
        print(data['store'])
        print(data['categories'])
        print(data['details'])
        break

dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together'])
{
    "main_category": null,
    "title": "Sure-Grip Zombie Wheels Low 59mm 4 Pack",
    "average_rating": 4.5,
    "rating_number": 84,
    "features": [
        "Pre-packaged in sets of 4",
        "Low profile 59mm x 38mm",
        "89a w/purple hub, 92a w/black hub, 95a w/red hub, 98a w/green hub",
        "Made in the U.S.A.",
        "Anodized Aluminum Hub"
    ],
    "description": [
        "All Zombie wheels are made in the USA. Zombie wheels feature anodized aluminum hubs for maximum durability and precise feel while maintaining rock solid stability. This allows our unique urethane compounds to deliver all your power to the floor. Choose the Zombie combination that fits your skating style and surface. Zombie Aluminum Core \u2013 Designed in house and manufactured using state of the 

### Examine Review Data
Tried to find one review of the above product. Relevant columns:
- title
- text

In [28]:
# Examine review data
data_folder = Path("../data/raw")
review_path = data_folder / "Sports_and_Outdoors.jsonl.gz"
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(review_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        if data.get("parent_asin") == 'B01HDXC8AG':
            print(json.dumps(data, indent=4))
            break

{
    "rating": 5.0,
    "title": "Excellent wheels",
    "text": "These replaced older wheels.  I mean from 80s old.  They are right height and solid.  The grip is a bit more than I want but that is no fault of wheels. I knew when I bought them they\u2019d be grippier but I didn\u2019t want to go less grippy.  I\u2019m very happy with performance.",
    "images": [],
    "asin": "B0157O33ES",
    "parent_asin": "B01HDXC8AG",
    "user_id": "AEHRHCKAPFO5RSX3VM73MZUXYJNA",
    "timestamp": 1520302057097,
    "helpful_vote": 4,
    "verified_purchase": true
}


### Retrieve the first 200 products for PoC

In [41]:
import pandas as pd

In [45]:
product_count = 1
product_list = list()
parent_asin_set = set()
threshold = 200
unwanted_keys = ['main_category', 'rating_number', 'images', 'videos', 'store', 'details', 'bought_together', 'subtitle', 'author']
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        data['features'] = ' '.join(data['features'])
        data['description'] = ' '.join(data['description'])
        for key in unwanted_keys:
            data.pop(key, None)
        product_list.append(data)
        parent_asin_set.add(data['parent_asin'])

        product_count += 1
        if product_count >= threshold:
            break

products_df = pd.DataFrame(product_list)
products_df.head()

,title,average_rating,features,description,price,categories,parent_asin
0,Sure-Grip Zombie Wheels Low 59mm 4 Pack,4.5,Pre-packaged in sets of 4 Low profile 59mm x 3...,All Zombie wheels are made in the USA. Zombie ...,55.00,"[Sports & Outdoors, Sports, Skates, Skateboard...",B01HDXC8AG
1,USGI Wet Weather Bag (Fоur Paсk),4.2,,Wet Wetaher bag US Military 4 pack,NaN,"[Sports & Outdoors, Sports, Boating & Sailing,...",B07R5BQ4YD
2,NHL San Jose Sharks Team Logo Post Earrings,4.5,100% Synthetic Imported Adorable Post Earrings...,Complete your game day outfit with these cute ...,18.99,"[Sports & Outdoors, Fan Shop, Jewelry & Watche...",B003K8GZ7G
3,Bont Skates - Prostar Purple Suede Professiona...,4.2,Roller Skates are not like your average shoe. ...,,209.00,"[Sports & Outdoors, Sports, Skates, Skateboard...",B08GC4GBWB
4,Team Golf Alamaba Crimson Tide Embroidered Tow...,5.0,Cotton Tri-fold golf towel is embroidered with...,Keep your clubs clean while supporting your fa...,NaN,"[Sports & Outdoors, Fan Shop, Sports Equipment...",B07BYV947H


In [ ]:
# Identify the corresponding review data

data_folder = Path("../data/raw")
review_path = data_folder / "Sports_and_Outdoors.jsonl.gz"
# Gemini's approach to reading .jsonl.gz without unzipping
with gzip.open(review_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Asked Gemini: how to print JSON in format
        if data.get("parent_asin") in parent_asin_set:
            print(json.dumps(data, indent=4))
            break
